In [2]:
# Standard Libraries
import os  # Operating system-specific functions
import cv2  # OpenCV library for computer vision tasks
import pandas as pd  # Pandas library for data manipulation and analysis

# External Libraries
import torch  # PyTorch deep learning framework
import torch.nn as nn  # Neural network modules, e.g., nn.Linear, nn.Conv2d, BatchNorm, Loss functions
import torch.nn.functional as F  # Functional operations in neural networks
import torch.optim as optim  # Optimization algorithms, e.g., SGD, Adam, etc.
import torchvision.transforms as transforms  # Image transformations for datasets
from torch.optim.lr_scheduler import StepLR  # Learning rate scheduler for optimization
from torch.utils.data import Dataset, DataLoader  # Utilities for dataset management and mini-batches

from skimage import io  # Image Input/Output operations for scikit-image library
# Custom Scripts
import pickle

# Hyperparameters
img_size = 64
batch_size = 1
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

import torch
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image
import pandas as pd
import os
import numpy as np

class SolarRadiance(Dataset):
    def __init__(self, root_dir, labels, transform):
        self.root_dir = root_dir
        self.labels = labels
        self.transform = transform
        # self.data = self.load_dataset()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        img_path = self.labels.iloc[index,0]
        target = self.labels.iloc[index,1]

        image = io.imread(img_path)
        image = cv2.imread(img_path)#, cv2.IMREAD_COLOR)
        image = cv2.resize(image, (img_size, img_size))

        if self.transform:
             image = self.transform(image)

        y_label = torch.tensor(target)

        return image, y_label

def load_dataset(root_dir):
        ds = pd.DataFrame()
        dates = os.listdir(root_dir)

        try:
            for date in dates:
                infrared_folder = os.path.join(root_dir, date, "infrared")
                pyranometer_folder = os.path.join(root_dir, date, "pyranometer")
                csv_path = os.path.join(pyranometer_folder, "{date}.csv".format(date=date))
                if not os.path.exists(csv_path):
                    print("Skipping date {date} because it does not have both infrared and pyranometer folders".format(date=date))
                    continue
                ds_temp = getDs(infrared_folder, csv_path)
                
                # append the dataframe in the final dataframe
                # ds = ds.append(ds_temp)
                ds = pd.concat([ds, ds_temp], ignore_index=True)

                ds['name'] = ds['name'].apply(lambda img: os.path.join(root_dir, date, 'infrared', img))
        except Exception as e:
            print(e)
        return ds

def getDs(path, labels):
    pyranometer = pd.read_csv(labels)
    images = os.listdir(path)
    
   
    #convert column 1 to int
    X = pyranometer.iloc[:,0].astype(int)

    #convert to image names
    pyranometer.iloc[:,0] = X.apply(lambda x: str(x) + 'IR.png')
    
    # Filter pyranometer DataFrame based on the 'x' column
    filtered_pyranometer = pyranometer[pyranometer.iloc[:,0].isin(images)]
    # Display the result
    filtered_pyranometer.columns = ['name', 'value']

    filtered_pyranometer = filtered_pyranometer.drop_duplicates(subset='name')
    return filtered_pyranometer


random_state = 42
torch.manual_seed(random_state)

root = '/Users/ifranrahmannijhum/Projects/MDFF/MDFF-Solar-Irradiance-Estimation/data.csv'

data = pd.read_csv(root)
# data = data[:50000]
data_len = len(data)
train_size = int(0.7 * data_len)
test_size = data_len - train_size

train_data = data[:train_size]
test_data = data[train_size:]

feature_dir = 'Features/'

os.mkdir(feature_dir) if not os.path.exists(feature_dir) else None

train_root  = 'root'
val_root = 'root'

# Import the required module
from torchvision.models import vgg11

# Load the pre-trained AlexNet model and move it to the device (GPU or CPU)
model = vgg11(pretrained=True).to(device)
model.eval()

import numpy as np
from sklearn.metrics import mean_squared_error

# Define a function to register forward hooks for feature extraction
def forward_hook_registration(total_layers, layers, outputs):

    # Define a function to append the output of each layer to the outputs list
    def append_output(module, input, output):
        outputs.append(output)

    # Create a list to store the hook handles
    hd = [None] * total_layers
    i = 0

    # Register forward hooks for each layer
    for layer_type, layer_indices in layers.items():
        for layer in layer_indices:
            if layer_type == 'conv':
                hd[i] = model.features[layer].register_forward_hook(append_output)
                i += 1
            elif layer_type == 'fc':
                hd[i] = model.classifier[layer].register_forward_hook(append_output)
                i += 1
    return hd, outputs

# Define a function to remove the registered hooks
def remove_hooks(hd, total_layers):
    for i in range(total_layers):
        hd[i].remove()

def pad_to_desired_size(original_tensor, tensor_to_pad):
    desired_size = original_tensor.size(2)
    # Calculate the amount of padding needed for each dimension
    padding_needed = desired_size - tensor_to_pad.size(2)
    mean_value = torch.mean(original_tensor)
    # Pad the tensor with zeros
    padded_tensor = F.pad(tensor_to_pad, (0, padding_needed, 0, padding_needed), mode='constant', value=mean_value)

    return padded_tensor


import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
import pandas as pd
import numpy as np

# create dir for saving extracted features
train_feature_path = feature_dir+'train/'

if not os.path.exists(train_feature_path):
  os.mkdir(train_feature_path)


layers = {'fc': [3]}
total_layers = len(layers['fc'])

def extract_train_features(idx, data, pca_conv):
        dataset = SolarRadiance(root_dir= train_root, labels=data, transform = transforms.Compose([transforms.ToTensor()]))

        # Create the train and validation data loaders
        train_loader = DataLoader(dataset, batch_size=1, shuffle=True, num_workers=0)

        train_df = []

        with torch.no_grad():
                for image_tensor, targets in tqdm(train_loader, desc="Extracting Training Features"):

                        targets = targets.to(device)

                        # Create the Gaussian pyramid to upscale the image
                        image_list = [image_tensor]

                        conv_features = []
                        for x, input in enumerate(image_list[::-1], 1):
                                input = input.to(device)
                                outputs = []

                                hd, outputs = forward_hook_registration(total_layers, layers, outputs)
                                _ = model(input.to(device))
                                remove_hooks(hd, total_layers)
                                final = outputs[0]

                        print(final.shape)

                        # Concatenate all feature vectors
                        combined_tr_tensor = np.concatenate((final.cpu().numpy(), targets.unsqueeze(1).cpu().numpy()), axis=1)
                        # append to the list
                        train_df.append(combined_tr_tensor)

        concated_train_df = np.vstack(train_df)

        np.save(train_feature_path + 'train_features_{}.npy'.format(idx), concated_train_df)
        
        pca_conv.partial_fit(concated_train_df[:, :-1])

        return pca_conv



from sklearn.decomposition import IncrementalPCA
import joblib

batch_size = 100
num_rows = len(train_data)

ipca_path = feature_dir+'ipca/'
if not os.path.exists(ipca_path):
        os.mkdir(ipca_path)

ipca_conv = IncrementalPCA(n_components=20, batch_size=batch_size)

i = 0
while i < num_rows:
        batch_filtered_pyranometer = train_data[i:i+batch_size]
        print("Processing rows {} to {}".format(i, i+batch_size))
        
        if os.path.exists(ipca_path+'ipca_conv{}.pkl'.format(i)):              
                # Load the PCA checkpoints
                joblib.load(ipca_path+'ipca_conv{}.pkl'.format(i))
                
        # Place your code here to process the batch of 100 rows
        ipca_conv = extract_train_features(i, batch_filtered_pyranometer, ipca_conv)
        
        # Save the PCA checkpoints
        joblib.dump(ipca_conv, ipca_path+'ipca_conv{}.pkl'.format(i))

        i += batch_size

# Save the final PCA models
joblib.dump(ipca_conv, ipca_path+'ipca_conv.pkl')



/Users/ifranrahmannijhum/miniforge3/envs/pytorch/lib/python3.8/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/ifranrahmannijhum/miniforge3/envs/pytorch/lib/python3.8/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG11_Weights.IMAGENET1K_V1`. You can also use `weights=VGG11_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Processing rows 0 to 100


Extracting Training Features:  10%|█         | 10/100 [00:00<00:00, 99.37it/s]

torch.Size([1, 4096])
torch.Size([1, 4096])
torch.Size([1, 4096])
torch.Size([1, 4096])
torch.Size([1, 4096])
torch.Size([1, 4096])
torch.Size([1, 4096])
torch.Size([1, 4096])
torch.Size([1, 4096])
torch.Size([1, 4096])
torch.Size([1, 4096])
torch.Size([1, 4096])
torch.Size([1, 4096])
torch.Size([1, 4096])
torch.Size([1, 4096])
torch.Size([1, 4096])
torch.Size([1, 4096])
torch.Size([1, 4096])
torch.Size([1, 4096])


Extracting Training Features:  21%|██        | 21/100 [00:00<00:00, 100.76it/s]

torch.Size([1, 4096])
torch.Size([1, 4096])


... (107 outputs left out)

torch.Size([1, 4096])
torch.Size([1, 4096])
torch.Size([1, 4096])
torch.Size([1, 4096])
torch.Size([1, 4096])
torch.Size([1, 4096])
torch.Size([1, 4096])
torch.Size([1, 4096])


Extracting Training Features:  65%|██████▌   | 65/100 [00:00<00:00, 89.91it/s]


torch.Size([1, 4096])


KeyboardInterrupt: 

In [ ]:
                        print(conv_features[1].view(1, -1).shape)
                        i2 = pad_to_desired_size(conv_features[0], conv_features[1])
                        # print(i1.shape, i2.shape)

                        # conv = torch.mean(torch.stack((conv1, conv2)), dim=0)
                        conv = torch.cat((i1, i2), dim=1)
                        conv = conv.view(len(targets), -1)
    
                        # Concatenate all feature vectors
                        combined_tr_tensor = np.concatenate((conv.cpu().numpy(), targets.unsqueeze(1).cpu().numpy()), axis=1)
                        print('i1 shape:', conv_features[0].shape)
                        print('i2 shape:', conv_features[1].shape)
                        print('conv shape:', conv.shape)
                        break
                        # append to the list
                        train_df.append(combined_tr_tensor)

        concated_train_df = np.vstack(train_df)

        np.save(train_feature_path + 'train_features_{}.npy'.format(idx), concated_train_df)
        
        pca_conv.partial_fit(concated_train_df[:, :-1])

        return pca_conv



from sklearn.decomposition import IncrementalPCA
import joblib

batch_size = 100
num_rows = len(train_data)

ipca_path = feature_dir+'ipca/'
if not os.path.exists(ipca_path):
        os.mkdir(ipca_path)

ipca_conv = IncrementalPCA(n_components=20, batch_size=batch_size)

i = 0
while i < num_rows:
        batch_filtered_pyranometer = train_data[i:i+batch_size]
        print("Processing rows {} to {}".format(i, i+batch_size))
        
        if os.path.exists(ipca_path+'ipca_conv{}.pkl'.format(i)):              
                # Load the PCA checkpoints
                joblib.load(ipca_path+'ipca_conv{}.pkl'.format(i))
                
        # Place your code here to process the batch of 100 rows
        ipca_conv = extract_train_features(i, batch_filtered_pyranometer, ipca_conv)
        
        break